### Modelo BERT como fábrica de embeddings:

In [49]:
phrases = [
    "Luffy quer se tornar o Rei dos Piratas",
    "Naruto sonha em ser Hokage da vila",
    "O anime de Attack on Titan tem reviravoltas incríveis",
    "Goku treina para superar seus limites em Dragon Ball",
    "O novo processador Intel tem 12 núcleos",
    "Minha placa de vídeo trava jogos em 4K",
    "O SSD deixou meu computador muito mais rápido",
    "Atualizei a memória RAM do notebook para 16GB",
    "Tive que trocar a placa de vídeo, então comprei uma RTX 5060"
]

# 0 = anime, 1 = tecnologia
rotulos = [0, 0, 0, 0, 1, 1, 1, 1, 1]

In [50]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "bert-base-multilingual-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
bert = AutoModel.from_pretrained(model_name)

for parameter in bert.parameters():
    parameter.requires_grad = False #Congelando os gradientes do bert para não aprender nada novo.




Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [51]:
def embedding_generation(phrase):

    inputs = tokenizer(phrase, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        output = bert(**inputs)
    # pega o embedding do token [CLS], que resume a frase inteira
    embedding_phrase = output.last_hidden_state.mean(dim=1)
    return embedding_phrase.squeeze()

In [52]:
embeddings = torch.stack([embedding_generation(f) for f in phrases])
rotulos_tensor = torch.tensor(rotulos, dtype=torch.float32).unsqueeze(1)

print(embeddings.shape)  # algo tipo torch.Size([8, 768]) -> 8 frases, 768 números cada
print(embeddings)

torch.Size([9, 768])
tensor([[-0.1381,  0.0806, -0.0744,  ...,  0.4240,  0.1332, -0.0047],
        [-0.2761,  0.0644,  0.0382,  ...,  0.1251,  0.0933, -0.3184],
        [-0.3327,  0.4130,  0.1623,  ...,  0.1215,  0.0645, -0.1201],
        ...,
        [-0.0997,  0.1083,  0.0533,  ...,  0.1012,  0.1328, -0.1362],
        [-0.2020,  0.1794,  0.0045,  ..., -0.1481, -0.2261, -0.3311],
        [-0.3436, -0.0064,  0.0237,  ..., -0.0134, -0.1221, -0.3477]])


# Meu classificador:

In [53]:
import torch.nn as nn

classificator = nn.Sequential(
    nn.Linear(768, 1),
    nn.Sigmoid()
)

loss_fn = nn.BCELoss()

optim = torch.optim.Adam(classificator.parameters(), lr=0.01)


In [54]:
for p in range(101):
    prev = classificator(embeddings)
    loss = loss_fn(prev, rotulos_tensor)

    optim.zero_grad()
    loss.backward()
    optim.step()

    if p % 50 == 0:

        print(f"Época {p}: loss = {loss.item():.4f}")
        for f , pr in zip(phrases,prev):
            print(f"Para a frase '{f}'. O valor é: {pr.item():.4f}")

Época 0: loss = 0.7267
Para a frase 'Luffy quer se tornar o Rei dos Piratas'. O valor é: 0.4205
Para a frase 'Naruto sonha em ser Hokage da vila'. O valor é: 0.3982
Para a frase 'O anime de Attack on Titan tem reviravoltas incríveis'. O valor é: 0.4396
Para a frase 'Goku treina para superar seus limites em Dragon Ball'. O valor é: 0.4445
Para a frase 'O novo processador Intel tem 12 núcleos'. O valor é: 0.4245
Para a frase 'Minha placa de vídeo trava jogos em 4K'. O valor é: 0.4393
Para a frase 'O SSD deixou meu computador muito mais rápido'. O valor é: 0.4148
Para a frase 'Atualizei a memória RAM do notebook para 16GB'. O valor é: 0.3967
Para a frase 'Tive que trocar a placa de vídeo, então comprei uma RTX 5060'. O valor é: 0.4335
Época 50: loss = 0.0021
Para a frase 'Luffy quer se tornar o Rei dos Piratas'. O valor é: 0.0018
Para a frase 'Naruto sonha em ser Hokage da vila'. O valor é: 0.0023
Para a frase 'O anime de Attack on Titan tem reviravoltas incríveis'. O valor é: 0.0031
Para

In [55]:
import torch.nn.functional as F

e1 = embedding_generation("Luffy quer se tornar o Rei dos Piratas")
e2 = embedding_generation("Naruto sonha em ser Hokage")
e3 = embedding_generation("O novo processador Intel tem 12 núcleos")

print("anime vs anime:", F.cosine_similarity(e1.unsqueeze(0), e2.unsqueeze(0)).item())
print("anime vs tech:", F.cosine_similarity(e1.unsqueeze(0), e3.unsqueeze(0)).item())

anime vs anime: 0.6761393547058105
anime vs tech: 0.6038002967834473


In [56]:
frase_teste = ["One Piece tem centenas de episódios emocionantes",
    "Comprei uma nova placa mãe para o setup"
]

for frase in frase_teste:
    embeddings = embedding_generation(frase).unsqueeze(0)
    with torch.no_grad():
        prob = classificator(embeddings).item()
    cat = "anime" if prob < 0.5 else "tech"
    print(f"'{frase}' | {cat}")


'One Piece tem centenas de episódios emocionantes' | anime
'Comprei uma nova placa mãe para o setup' | tech


In [75]:
frase = input("Digite uma frase sobre anime ou tech: ")
# frase = "Comprei uma nova placa mãe para o setup"
# embedding = embedding_generation(frase)
embedding = embedding_generation(frase).unsqueeze(0)
prob = classificator(embedding).item() #já pegar a probabilidade dentro do tensor gerado com .item()
categoria = "anime" if prob < 0.5 else "tech"

print(categoria)

tech
